# Align Development Notebook

This notebook is for interactive experiments with `vg_pipeline.align`. Use it to validate parsing, depth sampling, and back-projection logic before migrating changes into the main codebase.

In [1]:
import os
import sys
from pathlib import Path

root = Path('..').resolve()
sys.path.append(str(root))
print('Project root:', root)

from PIL import Image
from vg_pipeline.io import load_observation_npy
from vg_pipeline.align import (
    parse_align_result,
    parse_align_results_multi,
    sample_depth_median,
    deproject_pixel,
)
from vg_pipeline.providers import run_vg_inference
from vg_pipeline.prompting import build_align_prompt_multi
from grasp_server.align_grasp import (
    _pose_from_point_and_angle,
    build_align_grasp,
    _DEFAULT_WIDTH_M,
)
from grasp_server.grasp_selection import _rotation_to_quaternion_xyzw

print(
    'Imported project helpers: load_observation_npy, vg_pipeline.align, Gemini inference,\n'
    '_pose_from_point_and_angle, _rotation_to_quaternion_xyzw, build_align_grasp'
)

Project root: /Users/zitian/vla-grasp-server


Imported project helpers: load_observation_npy, vg_pipeline.align, Gemini inference,
_pose_from_point_and_angle, _rotation_to_quaternion_xyzw, build_align_grasp


## Part 1: Data Preparation
Load or create sample RGB, depth, and camera intrinsics for testing the alignment pipeline.

In [2]:
import numpy as np
import json

# Load sample RGBD data from captured images using the project loader
# (same helper the server uses: vg_pipeline.io.load_observation_npy)
sample_dir = root / 'rgbd_data' / 'captures' / '20260417_115700'
camera_data = load_observation_npy(sample_dir / 'camera_data.npy')

# Extract RGB, depth, and camera intrinsics
rgb_image = camera_data['rgb'].astype(np.uint8)
depth_map = camera_data['depth'].astype(np.float32)  # in meters
K = camera_data['K'].astype(np.float32)  # camera intrinsic matrix 3x3

print(f"RGB shape: {rgb_image.shape}")
print(f"Depth shape: {depth_map.shape}")
print(f"Camera intrinsics K:\n{K}")
print(f"Depth value range: [{np.nanmin(depth_map):.3f}, {np.nanmax(depth_map):.3f}] m")


RGB shape: (720, 1280, 3)
Depth shape: (720, 1280)
Camera intrinsics K:
[[921.9345    0.      638.43256]
 [  0.      922.3778  365.15457]
 [  0.        0.        1.     ]]
Depth value range: [0.000, 2.568] m


In [3]:
import re

# Gemini/VLM API key: prefer the environment, else read it from the zsh rc files (macOS).
def _api_key_from_zsh_rc():
    for rc in ('.zshrc', '.zshenv', '.zprofile'):
        path = Path.home() / rc
        if not path.exists():
            continue
        m = re.search(r'export\s+(GEMINI_API_KEY|GOOGLE_API_KEY)=["\']?([^"\'\n]+)', path.read_text())
        if m:
            return m.group(1), m.group(2)
    return None, None

api_key = os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY')
api_key_name = 'GEMINI_API_KEY' if os.environ.get('GEMINI_API_KEY') else 'GOOGLE_API_KEY' if api_key else None
if api_key is None:
    api_key_name, api_key = _api_key_from_zsh_rc()
    if api_key:
        os.environ[api_key_name] = api_key
        print(f'Loaded {api_key_name} from shell rc file.')

if api_key is None:
    raise RuntimeError('Set GEMINI_API_KEY or GOOGLE_API_KEY (e.g. export it in ~/.zshrc).')

print('Using API key from env var:', api_key_name)


Loaded GOOGLE_API_KEY from shell rc file.
Using API key from env var: GOOGLE_API_KEY


## Part 2: Parse VLM Output
The Vision Language Model (VLM) returns a JSON with normalized alignment point [y, x] (0-1000 scale) 
and gripper angle in degrees. We parse this and convert to full-resolution pixel coordinates.

In [4]:
# Real Gemini/VLM inference using project logic.
# The API key is loaded in Part 1 (api_key / api_key_name).

provider = 'gemini'
task_spec = 'the rail'
num_candidates = 1
prompt = build_align_prompt_multi(
    task_spec,
    w=rgb_image.shape[1],
    h=rgb_image.shape[0],
    num_candidates=num_candidates,
)

print('Using task spec:', task_spec)
print('Using provider:', provider)
print('Using API key from env var:', api_key_name)
print('Prompt preview:\n', prompt[:1000], '...\n')

raw_model_text = run_vg_inference(
    provider=provider,
    images=[Image.fromarray(rgb_image)],
    task_spec=task_spec,
    model_path='gemini-robotics-er-1.6-preview',
    num_candidates=num_candidates,
    api_key=api_key,
    openai_image_mime_types=['image/png'],
    gemini_image_mime_types=['image/png'],
    prompt=prompt,
)

print('Raw Gemini response:\n')
print(raw_model_text)

parsed_results = parse_align_results_multi(
    raw_model_text,
    canvas_h=rgb_image.shape[0],
    canvas_w=rgb_image.shape[1],
    rgb_h=rgb_image.shape[0],
)
if not parsed_results:
    raise RuntimeError('No align results returned from Gemini parse.')

align_result = parsed_results[0]
print(f'\nParsed Align Result:')
print(f'  Point (pixel coordinates [y, x]): {align_result.point_yx}')
print(f'  Angle (degrees): {align_result.angle_deg}')
print(f'  Width (meters): {align_result.width_m}')


Using task spec: the rail
Using provider: gemini
Using API key from env var: GOOGLE_API_KEY
Prompt preview:
 Role:
You are an expert in Embodied AI and robot vision. Your task is to analyze one RGB image and propose 1 DIVERSE candidate alignment/cut-in points for a robotic parallel-jaw gripper, ranked from most to least preferred.

Image description:
- You are given a single RGB image with size 1280 (width) x 720 (height) pixels.
- The image contains only color appearance information; no depth map is provided.
- The robot already has a separate depth sensor; you must only locate the 2D points and the in-image gripper orientations. The 3D distance is recovered later from depth.

Reasoning task:
1. Target identification: find the object/rail corresponding to "the rail" in the image.
2. Diversity planning (do this BEFORE choosing coordinates): mentally divide the target object into distinct spatial zones (e.g. upper / middle / lower section, left side / right side, narrow end / wide end).

Raw Gemini response:

```json
{
  "target": "the rail",
  "candidates": [
    {
      "rank": 1,
      "align_point": [375, 588],
      "gripper_angle_deg": 0,
      "reasoning": "Zone: middle section. The point is centered on the visible side of the object to ensure a stable grip. A horizontal gripper orientation (0 degrees) is chosen to close across the width of the object while maintaining maximum clearance from the table surface."
    }
  ]
}
```

Parsed Align Result:
  Point (pixel coordinates [y, x]): (270, 753)
  Angle (degrees): 0.0
  Width (meters): None


## Part 3: Sample Depth at Alignment Point
Extract the depth value at (or near) the alignment point. We take the median of valid depth values
in a 5×5 window around the point to be robust to noise and missing data.

In [5]:
v, u = align_result.point_yx  # pixel coordinates
print(f"Sampling depth at pixel ({v}, {u})")

# Sample depth using a 5×5 window around the point
depth_window = 5
z_m = sample_depth_median(depth_map, v, u, window=depth_window)

print(f"Sampled depth (median in {depth_window}×{depth_window} window): {z_m:.4f} m")
print(f"Depth in cm: {z_m * 100:.2f} cm")


Sampling depth at pixel (270, 753)
Sampled depth (median in 5×5 window): 0.5820 m
Depth in cm: 58.20 cm


## Part 4: Back-project 2D Pixel to 3D Camera Coordinates
Using the pinhole camera model and intrinsic matrix K, convert the 2D pixel + depth to a 3D point
in the camera frame: (X, Y, Z) where Z is along the optical axis (approach direction).

In [6]:
# Back-project pixel (u, v) with depth z to camera-frame 3D point
# Using pinhole camera model: [X, Y, Z] = [(u - cx) * Z / fx, (v - cy) * Z / fy, Z]

position_xyz = deproject_pixel(u, v, z_m, K)

print(f"Back-projection from pixel ({u}, {v}) with depth {z_m:.4f} m:")
print(f"  3D Position (camera frame):")
print(f"    X: {position_xyz[0]:.4f} m")
print(f"    Y: {position_xyz[1]:.4f} m")
print(f"    Z: {position_xyz[2]:.4f} m (approach direction)")
print(f"  Distance from camera: {np.linalg.norm(position_xyz):.4f} m")


Back-projection from pixel (753, 270) with depth 0.5820 m:
  3D Position (camera frame):
    X: 0.0723 m
    Y: -0.0600 m
    Z: 0.5820 m (approach direction)
  Distance from camera: 0.5895 m


## Part 5: Build 6-DoF Grasp Pose
From the 3D position and in-plane rotation angle, construct a 4×4 pose matrix with:
- X-axis (column 0): gripper closing direction (in image plane)
- Y-axis (column 1): lateral direction (cross product)
- Z-axis (column 2): approach direction (fixed to camera +Z)
- Position (column 3): the 3D point

In [7]:
# Build the 4×4 pose matrix from position and angle using the project helper
# (grasp_server.align_grasp._pose_from_point_and_angle) so the notebook exercises
# the exact production pose convention instead of re-deriving it:
#   - approach is fixed to +Z (camera optical axis)
#   - closing direction lies in the image plane, rotated by angle_deg
angle_deg = align_result.angle_deg
pose_4x4 = _pose_from_point_and_angle(position_xyz, angle_deg)

# pose columns: 0 = closing (in-plane @ angle), 1 = lateral, 2 = approach (+Z)
x_axis, y_axis, z_axis = pose_4x4[:3, 0], pose_4x4[:3, 1], pose_4x4[:3, 2]

print(f"4×4 Pose Matrix (camera frame):")
print(pose_4x4)
print(f"\nOrientation vectors:")
print(f"  X-axis (closing):  {x_axis}")
print(f"  Y-axis (lateral):  {y_axis}")
print(f"  Z-axis (approach): {z_axis}")


4×4 Pose Matrix (camera frame):
[[ 1.          0.          0.          0.07232429]
 [ 0.          1.          0.         -0.06004043]
 [ 0.          0.          1.          0.58200002]
 [ 0.          0.          0.          1.        ]]

Orientation vectors:
  X-axis (closing):  [1. 0. 0.]
  Y-axis (lateral):  [0. 1. 0.]
  Z-axis (approach): [0. 0. 1.]


## Part 6: Convert to Quaternion and Final Output
Convert the rotation matrix to a quaternion (xyzw format) for compatibility with ROS2 and downstream
processing. Assemble the final grasp dictionary with all 6-DoF information.

In [8]:
# Convert rotation matrix to quaternion (xyzw format) using the SAME converter the
# server uses (grasp_server.grasp_selection._rotation_to_quaternion_xyzw, Shepperd's
# method) rather than scipy — so this matches the production code path exactly.
R = pose_4x4[:3, :3]
qx, qy, qz, qw = _rotation_to_quaternion_xyzw(R)

print(f"Quaternion (xyzw): [{qx:.6f}, {qy:.6f}, {qz:.6f}, {qw:.6f}]")

# Assemble the final grasp dictionary by hand to document the output schema.
# NOTE: grasp_server.align_grasp.build_align_grasp() produces this exact dict in one
# call (see Part 7, which verifies the two paths agree field-for-field).
gripper_width = align_result.width_m if align_result.width_m is not None else _DEFAULT_WIDTH_M

grasp_dict = {
    "score": 1.0,
    "model_confidence": None,
    "pose_4x4": pose_4x4.tolist(),
    "position_xyz": position_xyz.tolist(),
    "quaternion_xyzw": [qx, qy, qz, qw],
    "width_m": float(gripper_width),
    "approach_dir_xyz": pose_4x4[:3, 2].tolist(),
    "source": {
        "candidate_index": 0,
        "segment_id": 0,
        "grasp_index": 0,
        "predictions_npz": None,
    },
}

print("\n=== FINAL OUTPUT: 6-DoF Grasp ===")
print(json.dumps(grasp_dict, indent=2, default=str))


Quaternion (xyzw): [0.000000, 0.000000, 0.000000, 1.000000]

=== FINAL OUTPUT: 6-DoF Grasp ===
{
  "score": 1.0,
  "model_confidence": null,
  "pose_4x4": [
    [
      1.0,
      0.0,
      0.0,
      0.07232428509296399
    ],
    [
      0.0,
      1.0,
      0.0,
      -0.060040432248501065
    ],
    [
      0.0,
      0.0,
      1.0,
      0.5820000171661377
    ],
    [
      0.0,
      0.0,
      0.0,
      1.0
    ]
  ],
  "position_xyz": [
    0.07232428509296399,
    -0.060040432248501065,
    0.5820000171661377
  ],
  "quaternion_xyzw": [
    0.0,
    0.0,
    0.0,
    1.0
  ],
  "width_m": 0.05,
  "approach_dir_xyz": [
    0.0,
    0.0,
    1.0
  ],
  "source": {
    "candidate_index": 0,
    "segment_id": 0,
    "grasp_index": 0,
    "predictions_npz": null
  }
}


## Part 7: Verify End-to-End Pipeline
Run the complete pipeline using the `build_align_grasp()` function to verify it produces the same result.

In [9]:
# Use project helper to build the final grasp(s) from the aligned point
grasps_list = build_align_grasp(
    depth_map,
    K,
    point_yx=align_result.point_yx,
    angle_deg=align_result.angle_deg,
    width_m=align_result.width_m,
    depth_window=5,
)

print("=== End-to-End Project Helper Result ===")
print(f"Number of grasps: {len(grasps_list)}")
grasp_output = grasps_list[0]
print("\nGrasp dictionary:")
print(json.dumps(grasp_output, indent=2, default=str))

# Compare with the manual build path if present
if 'grasp_dict' in globals():
    print("\n=== Verification ===")
    print(f"Position matches: {np.allclose(grasp_output['position_xyz'], grasp_dict['position_xyz'])}")
    print(f"Quaternion matches: {np.allclose(grasp_output['quaternion_xyzw'], grasp_dict['quaternion_xyzw'])}")
    print(f"Width matches: {grasp_output['width_m'] == grasp_dict['width_m']}")
    print("\n✓ Complete 2D-to-6D alignment pipeline verified!")
else:
    print("No manual grasp_dict available for direct comparison.")


=== End-to-End Project Helper Result ===
Number of grasps: 1

Grasp dictionary:
{
  "score": 1.0,
  "model_confidence": null,
  "pose_4x4": [
    [
      1.0,
      0.0,
      0.0,
      0.07232428509296399
    ],
    [
      0.0,
      1.0,
      0.0,
      -0.060040432248501065
    ],
    [
      0.0,
      0.0,
      1.0,
      0.5820000171661377
    ],
    [
      0.0,
      0.0,
      0.0,
      1.0
    ]
  ],
  "position_xyz": [
    0.07232428509296399,
    -0.060040432248501065,
    0.5820000171661377
  ],
  "quaternion_xyzw": [
    0.0,
    0.0,
    0.0,
    1.0
  ],
  "width_m": 0.05,
  "approach_dir_xyz": [
    0.0,
    0.0,
    1.0
  ],
  "source": {
    "candidate_index": 0,
    "segment_id": 0,
    "grasp_index": 0,
    "predictions_npz": null
  }
}

=== Verification ===
Position matches: True
Quaternion matches: True
Width matches: True

✓ Complete 2D-to-6D alignment pipeline verified!
